# Vehicle Detection and Classification using YOLOv8
**Team Members:** Aragya Goyal and Olivia Sobek  
**Dataset:** [Link to Dataset](https://www.kaggle.com/datasets/barkataliarbab/vehicles-openimages-dataset-416416/data)  
**Model:** YOLOv8s - Finetuned via transfer learning

## 1. Setup and Installation

In [ ]:
# Import Packages
import os
import shutil
import kagglehub
from collections import Counter
import matplotlib.pyplot as plt

# Define Globals
IMG_W, IMG_H = 416, 416
CLASS_NAMES = {0: "Ambulance", 1: "Bus", 2: "Car", 3: "Motorcycle", 4: "Truck"}

## 2. Download Dataset from Kaggle

In [26]:
# Use kagglehub to download the dataset
cache_path = kagglehub.dataset_download('barkataliarbab/vehicles-openimages-dataset-416416')
print("Downloaded to:", cache_path)

Downloaded to: C:\Users\agoyal1642\.cache\kagglehub\datasets\barkataliarbab\vehicles-openimages-dataset-416416\versions\1


## 3. Copy files to local workspace and make YOLO labels

In [27]:
# Define dataset path and splits
dataset_root = "./dataset"
split_names = ["train", "valid", "test"]

# Copy the raw data
if os.path.exists(dataset_root):
    shutil.rmtree(dataset_root)

for split in split_names:
    src = os.path.join(cache_path, split)
    dst = os.path.join(dataset_root, split)
    shutil.copytree(src, dst)
    print(f"Copied {split} images: {len(os.listdir(dst))} files")

# Print break
print()

# Rename the files and reorganize
for split in split_names:
    # Get annotation file path
    split_dir = os.path.join(dataset_root, split)
    ann_path = os.path.join(split_dir, "_annotations.txt")

    # Create image and label folders
    img_dir = os.path.join(split_dir, "images")
    lbl_dir = os.path.join(split_dir, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    # Parse the annotation file
    annotations = {}
    with open(ann_path, "r") as ann_file:
        for line in ann_file:
            # Get file names and bounding boxes with classifications
            parts = line.strip().split()
            
            # Parse through parts
            filename = parts[0]
            boxes = []
            for box_str in parts[1:]:
                x1, y1, x2, y2, cls_id = map(int, box_str.split(","))
                boxes.append((x1, y1, x2, y2, cls_id))
            
            # Add to dictionary
            annotations[filename] = boxes
    
    # Rename, move, and create labels
    count = 0
    for idx, (old_filename, boxes) in enumerate(sorted(annotations.items()), start=1):
        # Get old image path
        old_img_path = os.path.join(split_dir, old_filename)

        # Create new name for file based on part of the split and the id
        ext = os.path.splitext(old_filename)[1]
        new_basename = f"{split}_{idx:04d}"
        new_img_name = new_basename + ext

        # Move image into new directory with new name
        os.rename(old_img_path, os.path.join(img_dir, new_img_name))

        # Convert to YOLO format and write label file
        yolo_lines = []
        for (x1, y1, x2, y2, cls_id) in boxes:
            cx = ((x1 + x2) / 2) / IMG_W
            cy = ((y1 + y2) / 2) / IMG_H
            w = (x2 - x1) / IMG_W
            h = (y2 - y1) / IMG_H
            yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        with open(os.path.join(lbl_dir, new_basename + ".txt"), "w") as f:
            f.write("\n".join(yolo_lines) + "\n")

        count += 1

    print(f"{split}: {count} images -> images/ + labels/")

Copied train images: 880 files
Copied valid images: 252 files
Copied test images: 128 files

train: 878 images -> images/ + labels/
valid: 250 images -> images/ + labels/
test: 126 images -> images/ + labels/


## 4. Class Distribution Analysis

### Remove Dataset

In [ ]:
shutil.rmtree(dataset_root, ignore_errors=True)
print("Removed data locally")

shutil.rmtree(cache_path, ignore_errors=True)
print("Removed data from cache")